In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql import Window

from datetime import datetime, timezone

In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("batch_id", "2025-01-15")
dbutils.widgets.dropdown("run_mode", "initial", ["initial", "incremental"])
dbutils.widgets.text("job_run_id", "")

environment = dbutils.widgets.get("environment")
batch_id = dbutils.widgets.get("batch_id")
run_mode = dbutils.widgets.get("run_mode")


pipeline_name = "education_qa_pipeline"

job_run_id = dbutils.widgets.get("job_run_id")

if job_run_id:
    run_id = f"{environment}_education_qa_pipeline_{batch_id}_{run_mode}_job_{job_run_id}"
else:
    run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{environment}_{pipeline_name}_{batch_id}_{run_mode}_{run_timestamp}"

if run_mode == "initial":
    write_mode = "overwrite"
elif run_mode == "incremental":
    write_mode = "append"
else:
    raise ValueError(f"Unsupported run_mode: {run_mode}")

catalog = "dbw_edu_qa_dev"
storage_account = "steduqadblakehouse"
container = "education-data-lake"

lake_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

print(f"environment: {environment}")
print(f"batch_id: {batch_id}")
print(f"run_mode: {run_mode}")
print(f"write_mode: {write_mode}")
print(f"job_run_id: {job_run_id}")
print(f"run_id: {run_id}")

environment: dev
batch_id: 2026-01-15
run_mode: incremental
write_mode: append
job_run_id: 
run_id: dev_education_qa_pipeline_2026-01-15_incremental_20260608T045106Z


In [0]:
bronze_tables = [
    "schools",
    "students",
    "attendance",
    "assessment_results",
    "school_events"
]

for table_name in bronze_tables:
    print(f"\n=== {catalog}.bronze.{table_name} ===")
    spark.table(f"{catalog}.bronze.{table_name}").printSchema()


=== dbw_edu_qa_dev.bronze.schools ===
root
 |-- school_id: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- school_type: string (nullable = true)
 |-- open_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- run_id: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)


=== dbw_edu_qa_dev.bronze.students ===
root
 |-- student_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- year_level: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- enrolment_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- run_id: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- bronze_record_id

In [0]:
# Idempotency guard for incremental Silver loads.
# If the same batch is rerun, remove existing Silver rows for that batch before appending replacement rows.

silver_target_tables = [
    "schools",
    "students",
    "attendance",
    "assessment_results",
    "school_events"
]

def table_exists(schema_name, table_name):
    return (
        spark.sql(f"SHOW TABLES IN {catalog}.{schema_name} LIKE '{table_name}'")
        .count() > 0
    )

if run_mode == "incremental":
    for table_name in silver_target_tables:
        if table_exists("silver", table_name):
            spark.sql(
                f"""
                DELETE FROM {catalog}.silver.{table_name}
                WHERE batch_id = '{batch_id}'
                """
            )
            print(f"Deleted existing Silver rows for batch_id={batch_id} from silver.{table_name}")
        else:
            print(f"Skipped delete because silver.{table_name} does not exist yet")


Deleted existing Silver rows for batch_id=2026-01-15 from silver.schools
Deleted existing Silver rows for batch_id=2026-01-15 from silver.students
Deleted existing Silver rows for batch_id=2026-01-15 from silver.attendance
Deleted existing Silver rows for batch_id=2026-01-15 from silver.assessment_results
Deleted existing Silver rows for batch_id=2026-01-15 from silver.school_events


In [0]:
def dedupe_latest_bronze_batch(df, business_key_columns):
    """Keep one latest Bronze row per batch and business key."""
    dedupe_window = (
        Window
        .partitionBy(["batch_id"] + business_key_columns)
        .orderBy(
            F.col("load_timestamp").desc(),
            F.col("run_id").desc(),
            F.col("bronze_record_id").desc()
        )
    )

    return (
        df
        .withColumn("silver_dedupe_row_number", F.row_number().over(dedupe_window))
        .filter(F.col("silver_dedupe_row_number") == 1)
        .drop("silver_dedupe_row_number")
    )

### silver schema

In [0]:
# silver.schools

target_path = f"{lake_root}/silver/schools"


schools_silver_df = (
    dedupe_latest_bronze_batch(
        spark.table(f"{catalog}.bronze.schools")
        .filter(F.col("batch_id") == batch_id),
        ["school_id"]
    )
    .select(
        F.trim(F.col("school_id")).alias("school_id"),
        F.trim(F.col("school_name")).alias("school_name"),
        F.trim(F.col("region")).alias("region"),
        F.trim(F.col("school_type")).alias("school_type"),
        F.to_date(F.col("open_date"), "yyyy-MM-dd").alias("open_date"),
        F.trim(F.col("status")).alias("status"),
        F.col("batch_id"),
        F.col("run_id"),
        F.col("load_timestamp"),
        F.col("source_file_name"),
        F.col("bronze_record_id")
    )
    .withColumn("silver_load_timestamp", F.current_timestamp())
)

writer = (
    schools_silver_df.write
    .format("delta")
    .mode(write_mode)
    .option("path", target_path)
)

if write_mode == "overwrite":
    writer = writer.option("overwriteSchema", "true").partitionBy("batch_id")
    
writer.saveAsTable(f"{catalog}.silver.schools")

display(
    spark.table(f"{catalog}.silver.schools")
    .orderBy("batch_id", "school_id")
    .limit(20)
)

school_id,school_name,region,school_type,open_date,status,batch_id,run_id,load_timestamp,source_file_name,bronze_record_id,silver_load_timestamp
SCH001,ACT Education School 001,North Canberra,High School,1994-09-04,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,826b4c348e2a4e7ae17b4ec5aa3e2922356330474ae6284a643fad4b0d7d7840,2026-06-08T04:19:14.898Z
SCH002,ACT Education School 002,North Canberra,Primary,2018-12-05,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,4cab647b1b40c87e53c8d0984857e81aae654daf25a23c726a649cbe422afab1,2026-06-08T04:19:14.898Z
SCH003,ACT Education School 003,North Canberra,Primary,1978-05-29,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,3d47d40e2a8a1abd2c7fbc1e1d151d58a21c8889a17c14da327ff7cf4ee0b9ce,2026-06-08T04:19:14.898Z
SCH004,ACT Education School 004,North Canberra,Primary,1987-11-03,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,68767674ffe39a43413cfb4eff0bc5d8998f204dff6f49eb60752e0bf9f6b280,2026-06-08T04:19:14.898Z
SCH005,ACT Education School 005,Tuggeranong,High School,1989-10-11,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,0e9f0c92559cd2023bc52f670174b9b937694ab8d003cebd7cdfb5f33b226e60,2026-06-08T04:19:14.898Z
SCH006,ACT Education School 006,Weston Creek,Primary,1970-08-01,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,46e3e72dd369db542790ea6f2c35a312bccec9a125b6c64b7c15636016662ba8,2026-06-08T04:19:14.898Z
SCH007,ACT Education School 007,Tuggeranong,Primary,2000-07-11,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dbd5b5ac5579f4881040869265809b392303fce0d5c26c80cacebec7dd4ba996,2026-06-08T04:19:14.898Z
SCH008,ACT Education School 008,Weston Creek,Primary,2000-03-13,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,f9e431e05e92e80cac05eba5d814de3577c5bd41c93945675a86393a4e24a410,2026-06-08T04:19:14.898Z
SCH009,ACT Education School 009,Belconnen,Primary,2000-11-09,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,b61e878ea6ab6525800223d5786827bc8c84cf060c55422df105f29b6c77a607,2026-06-08T04:19:14.898Z
SCH010,ACT Education School 010,Woden,College,2011-03-21,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:44.138Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,36ec29534cf6d7e3c901ea5263262fc094da80dc9f8b239b43935a1c19a37072,2026-06-08T04:19:14.898Z


In [0]:
spark.table(f"{catalog}.silver.schools").printSchema()

(
    spark.table(f"{catalog}.silver.schools")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
    .show()
)

root
 |-- school_id: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- school_type: string (nullable = true)
 |-- open_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+----------+-----+
|  batch_id|count|
+----------+-----+
|2025-01-15|   50|
|2026-01-15|   51|
+----------+-----+



In [0]:
# silver.students

target_path = f"{lake_root}/silver/students"

students_silver_df = (
    dedupe_latest_bronze_batch(
        spark.table(f"{catalog}.bronze.students")
        .filter(F.col("batch_id") == batch_id),
        ["student_id"]
    )
    .select(
        F.trim(F.col("student_id")).alias("student_id"),
        F.trim(F.col("school_id")).alias("school_id"),
        F.col("year_level").cast("int").alias("year_level"),
        F.trim(F.col("gender")).alias("gender"),
        F.to_date(F.col("enrolment_date"), "yyyy-MM-dd").alias("enrolment_date"),
        F.trim(F.col("status")).alias("status"),
        F.col("batch_id"),
        F.col("run_id"),
        F.col("load_timestamp"),
        F.col("source_file_name"),
        F.col("bronze_record_id")
    )
    .withColumn("silver_load_timestamp", F.current_timestamp())
)

writer = (
    students_silver_df.write
    .format("delta")
    .mode(write_mode)
    .option("path", target_path)
)

if write_mode == "overwrite":
    writer = writer.option("overwriteSchema", "true").partitionBy("batch_id")
    
writer.saveAsTable(f"{catalog}.silver.students")

display(
    spark.table(f"{catalog}.silver.students")
    .orderBy("batch_id", "student_id")
    .limit(20)
)

student_id,school_id,year_level,gender,enrolment_date,status,batch_id,run_id,load_timestamp,source_file_name,bronze_record_id,silver_load_timestamp
null,SCH001,8,Non-specified,2024-03-01,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,7f2e5ca1848c860a58bd2f2ea6e75fb0d303764760132bdfd75d2c47ef5d9317,2026-06-08T04:19:26.890Z
STU000001,SCH009,5,Male,2024-05-27,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,31a534e437f1b04c2166006da3c72624fddc59104eb5210ea12e914b521aa04a,2026-06-08T04:19:26.890Z
STU000002,SCH016,6,Female,2022-07-26,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,a162ee512cbd46877cb6618f8db49a704441fedac974634616a0ea93daa36e9b,2026-06-08T04:19:26.890Z
STU000003,SCH007,5,Female,2022-10-01,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,5a13ccc6f7c02199eb0d9ec77fb1f1e13e7615e84e0c03011e45aac65f1f62b7,2026-06-08T04:19:26.890Z
STU000004,SCH047,0,Male,2019-02-08,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,24fbd7aa1fc89b600d6487d7a077fbdc44fe847677fe78711f6d074f17a80ae5,2026-06-08T04:19:26.890Z
STU000005,SCH047,2,Male,2019-03-24,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,de5970124ee4e01b0e10c652cc6129e197a7eb5978c43f27f91cc2ca6b765095,2026-06-08T04:19:26.890Z
STU000006,SCH013,10,Female,2020-01-22,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,eb78a823cbdb001e1fe234de7713385fbdac44ea9ac12d8d6f12fd0d69ee2928,2026-06-08T04:19:26.890Z
STU000007,SCH016,6,Male,2022-12-21,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,3e748b020044eebad3d070bae9646fc579abdfc2c3b615554a3378164971bb3f,2026-06-08T04:19:26.890Z
STU000008,SCH036,0,Female,2024-01-24,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,b221b6cc9cc7ec8bd6719a6f7536ee198374aca1b3e232287d338c93f864b4ac,2026-06-08T04:19:26.890Z
STU000009,SCH006,6,Male,2019-11-13,Active,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:47.490Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/students/batch_id=2025-01-15/students.csv,2d67f0f53fda53a8bf4333d18c4da4586f6c15d558a05d7378d4bc5510327773,2026-06-08T04:19:26.890Z


In [0]:
spark.table(f"{catalog}.silver.students").printSchema()

(
    spark.table(f"{catalog}.silver.students")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
    .show()
)

root
 |-- student_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- year_level: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- enrolment_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+----------+-----+
|  batch_id|count|
+----------+-----+
|2025-01-15|10002|
|2026-01-15|10502|
+----------+-----+



In [0]:
# silver.attendance

target_path = f"{lake_root}/silver/attendance"

attendance_silver_df = (
    dedupe_latest_bronze_batch(
        spark.table(f"{catalog}.bronze.attendance")
        .filter(F.col("batch_id") == batch_id),
        ["attendance_id"]
    )
    .select(
        F.trim(F.col("attendance_id")).alias("attendance_id"),
        F.trim(F.col("student_id")).alias("student_id"),
        F.trim(F.col("school_id")).alias("school_id"),
        F.to_date(F.col("attendance_month"), "yyyy-MM-dd").alias("attendance_month"),
        F.col("possible_days").cast("int").alias("possible_days"),
        F.col("attended_days").cast("int").alias("attended_days"),
        F.trim(F.col("absence_reason")).alias("absence_reason"),
        F.col("batch_id"),
        F.col("run_id"),
        F.col("load_timestamp"),
        F.col("source_file_name"),
        F.col("bronze_record_id")
    )
    .withColumn("silver_load_timestamp", F.current_timestamp())
)

writer = (
    attendance_silver_df.write
    .format("delta")
    .mode(write_mode)
    .option("path", target_path)
)

if write_mode == "overwrite":
    writer = writer.option("overwriteSchema", "true").partitionBy("batch_id")
    
writer.saveAsTable(f"{catalog}.silver.attendance")

display(
    spark.table(f"{catalog}.silver.attendance")
    .orderBy("batch_id", "attendance_id")
    .limit(20)
)

attendance_id,student_id,school_id,attendance_month,possible_days,attended_days,absence_reason,batch_id,run_id,load_timestamp,source_file_name,bronze_record_id,silver_load_timestamp
ATT00000001,STU000001,SCH009,2024-01-01,0,0,null,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,93be66dfc2bdf2bfd27a27328aa6682c5c707febe96d285d545e671dab39fce3,2026-06-08T04:19:30.751Z
ATT00000002,STU000001,SCH009,2024-02-01,18,16,Medical,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,eee85fd6a3d5c9969038efbde90b8d523eb19b963f7cc591c035219ded024875,2026-06-08T04:19:30.751Z
ATT00000003,STU000001,SCH009,2024-03-01,18,17,Medical,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,550161692f3467204c282eb94681dda1df18faaa7158f9aad90e5d24bfff869c,2026-06-08T04:19:30.751Z
ATT00000004,STU000001,SCH009,2024-04-01,22,21,Other,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,b8706cabc0a45fc3bd93d208f327c1421baca9a8ece79f6e85a44eb53d11dc9d,2026-06-08T04:19:30.751Z
ATT00000005,STU000001,SCH009,2024-05-01,18,17,Unauthorised,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,15e1898585a2471c508006f5eed948c2cb88dbc63ff83165c9a0c08c0c1b20a6,2026-06-08T04:19:30.751Z
ATT00000006,STU000001,SCH009,2024-06-01,21,17,Other,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,ee254e22e51f7fef2ca987c9aa4e25c419ed43b138234401cce8fb257b50e5b4,2026-06-08T04:19:30.751Z
ATT00000007,STU000001,SCH009,2024-07-01,22,17,Illness,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,984ed9de3289d84410d8273fafa1312f3675b43e926bca1871b5b3dda8f5ea7e,2026-06-08T04:19:30.751Z
ATT00000008,STU000001,SCH009,2024-08-01,18,14,Unauthorised,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,2cd1cf671c7e7a49a10ef9c8e512a04f52a26ab9e8cc58b2a729558da58e63fa,2026-06-08T04:19:30.751Z
ATT00000009,STU000001,SCH009,2024-09-01,22,19,Medical,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,21f161e91fcb987cf6fc3aef41a1cf285f6d5e0cbda50c47caa06925c96186ab,2026-06-08T04:19:30.751Z
ATT00000010,STU000001,SCH009,2024-10-01,21,21,null,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:51.795Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/attendance/batch_id=2025-01-15/attendance.csv,9f27f090eebe069ea4b01bd255fe9647f24761f9bf60c5088c39f5f841b22c2b,2026-06-08T04:19:30.751Z


In [0]:
spark.table(f"{catalog}.silver.attendance").printSchema()

(
    spark.table(f"{catalog}.silver.attendance")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
    .show()
)

root
 |-- attendance_id: string (nullable = true)
 |-- student_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- attendance_month: date (nullable = true)
 |-- possible_days: integer (nullable = true)
 |-- attended_days: integer (nullable = true)
 |-- absence_reason: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+----------+------+
|  batch_id| count|
+----------+------+
|2025-01-15|120003|
|2026-01-15| 96939|
+----------+------+



In [0]:
# silver.assessment_results

target_path = f"{lake_root}/silver/assessment_results"

assessment_results_silver_df  = (
    dedupe_latest_bronze_batch(
        spark.table(f"{catalog}.bronze.assessment_results")
        .filter(F.col("batch_id") == batch_id),
        ["assessment_id"]
    )
    .select(
        F.trim(F.col("assessment_id")).alias("assessment_id"),
        F.trim(F.col("student_id")).alias("student_id"),
        F.trim(F.col("school_id")).alias("school_id"),
        F.col("assessment_year").cast("int").alias("assessment_year"),
        F.trim(F.col("domain")).alias("domain"),
        F.col("score").cast("int").alias("score"),
        F.trim(F.col("proficiency_band")).alias("proficiency_band"),
        F.col("batch_id"),
        F.col("run_id"),
        F.col("load_timestamp"),
        F.col("source_file_name"),
        F.col("bronze_record_id")
    )
    .withColumn("silver_load_timestamp", F.current_timestamp())
)

writer = (
    assessment_results_silver_df .write
    .format("delta")
    .mode(write_mode)
    .option("path", target_path)
)

if write_mode == "overwrite":
    writer = writer.option("overwriteSchema", "true").partitionBy("batch_id")
    
writer.saveAsTable(f"{catalog}.silver.assessment_results")

display(
    spark.table(f"{catalog}.silver.assessment_results")
    .orderBy("batch_id", "assessment_id")
    .limit(20)
)

assessment_id,student_id,school_id,assessment_year,domain,score,proficiency_band,batch_id,run_id,load_timestamp,source_file_name,bronze_record_id,silver_load_timestamp
ASM00000001,STU000001,SCH009,2024,Reading,460,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,24a164f3415368a56f2d6b3ab0680c9e4c501c3af7fa4e72114384d76de1696a,2026-06-08T04:19:34.880Z
ASM00000002,STU000001,SCH009,2024,Numeracy,397,Low,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,7c26d9ad640a5d59b827897114cb7678d09c2286c6058790088ce15707dd22a4,2026-06-08T04:19:34.880Z
ASM00000003,STU000001,SCH009,2024,Writing,365,Low,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,3ffb6670499b424631575c0e5d729d8792d72fff68497907219e49d5a7f66ec9,2026-06-08T04:19:34.880Z
ASM00000004,STU000002,SCH016,2024,Reading,449,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,6055efa52f9c34f7448f3e9e8d55ca6e10575d80c7b05a990ac3a4fc20159199,2026-06-08T04:19:34.880Z
ASM00000005,STU000002,SCH016,2024,Numeracy,449,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,2a33724e8d7fdf9fdc8bdde2a6c7a32bcd680fc17bec049f03b27dcc6d9cd379,2026-06-08T04:19:34.880Z
ASM00000006,STU000002,SCH016,2024,Writing,419,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,da3bdd358da7a8e324cdaea9047eca4aa7538c4b31292737e6015280ac21196e,2026-06-08T04:19:34.880Z
ASM00000007,STU000003,SCH007,2024,Reading,447,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,2160712e2306e2335a61fa857b8d386fddddba4876688f1f39407def190a2bca,2026-06-08T04:19:34.880Z
ASM00000008,STU000003,SCH007,2024,Numeracy,467,Medium,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,e731beecea44a2ba96001108b6e56f44d45abab05323224ace41449ba5f810e2,2026-06-08T04:19:34.880Z
ASM00000009,STU000003,SCH007,2024,Writing,399,Low,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,3c2b442336ca8d9f7cbf8078f8bdca5f1bba389187599fe973136c99950b4a5f,2026-06-08T04:19:34.880Z
ASM00000010,STU000004,SCH047,2024,Reading,320,Low,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:17:56.594Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/assessment_results/batch_id=2025-01-15/assessment_results.csv,c2529c9ed0def970ad917c48bf980aa57e44556c7fc27ef00614317c9a5675be,2026-06-08T04:19:34.880Z


In [0]:
spark.table(f"{catalog}.silver.assessment_results").printSchema()

(
    spark.table(f"{catalog}.silver.assessment_results")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
    .show()
)

root
 |-- assessment_id: string (nullable = true)
 |-- student_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- assessment_year: integer (nullable = true)
 |-- domain: string (nullable = true)
 |-- score: integer (nullable = true)
 |-- proficiency_band: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+----------+-----+
|  batch_id|count|
+----------+-----+
|2025-01-15|30001|
|2026-01-15|24235|
+----------+-----+



In [0]:
# silver.school_events

target_path = f"{lake_root}/silver/school_events"

school_events_silver_df = (
    dedupe_latest_bronze_batch(
        spark.table(f"{catalog}.bronze.school_events")
        .filter(F.col("batch_id") == batch_id),
        ["event_id"]
    )
    .select(
        F.trim(F.col("event_id")).alias("event_id"),
        F.trim(F.col("school_id")).alias("school_id"),
        F.to_date(F.col("event_date"), "yyyy-MM-dd").alias("event_date"),
        F.trim(F.col("event_type")).alias("event_type"),
        F.trim(F.col("description")).alias("description"),
        F.col("batch_id"),
        F.col("run_id"),
        F.col("load_timestamp"),
        F.col("source_file_name"),
        F.col("bronze_record_id")
    )
    .withColumn("silver_load_timestamp", F.current_timestamp())
)

writer = (
    school_events_silver_df.write
    .format("delta")
    .mode(write_mode)
    .option("path", target_path)
)

if write_mode == "overwrite":
    writer = writer.option("overwriteSchema", "true").partitionBy("batch_id")

writer.saveAsTable(f"{catalog}.silver.school_events")

display(
    spark.table(f"{catalog}.silver.school_events")
    .orderBy("batch_id", "event_id")
    .limit(20)
)

event_id,school_id,event_date,event_type,description,batch_id,run_id,load_timestamp,source_file_name,bronze_record_id,silver_load_timestamp
EVT00001,SCH001,2024-12-12,Wellbeing program,School-based wellbeing program supporting student engagement,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,fd210eae8e15c8276d16f01c693938297aefe6c347f590798383566cd2a2aa2e,2026-06-08T04:19:38.275Z
EVT00002,SCH002,2024-07-26,Wellbeing program,School-based wellbeing program supporting student engagement,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,c2f20ab1e8a0aa96b5a2b71c4ceff5148b4fc9a0ca3d7befbc7aefb85bca3ba8,2026-06-08T04:19:38.275Z
EVT00003,SCH002,2024-03-22,Attendance campaign,Campaign to improve student attendance and reduce unexplained absences,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,9c1d5d7144ecaf65eedd13fa44d4acbdd77e2288aac45e9b4dd03a745ede37ed,2026-06-08T04:19:38.275Z
EVT00004,SCH002,2024-05-24,Wellbeing program,School-based wellbeing program supporting student engagement,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,446b704961967047df0a774e4e7623c53c79b05992bdc33c23028051507fce2a,2026-06-08T04:19:38.275Z
EVT00005,SCH003,2024-04-13,Wellbeing program,School-based wellbeing program supporting student engagement,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,cc4c3a6dd5cb7f2a05d737623a052310bcb1e4b7590de19e663145e196ba01ae,2026-06-08T04:19:38.275Z
EVT00006,SCH004,2024-10-25,Attendance campaign,Campaign to improve student attendance and reduce unexplained absences,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,a999d7dc4ea43a42751bc0f8c3709eb27a40f0d11ca92d27c81ecad36f24a8a9,2026-06-08T04:19:38.275Z
EVT00007,SCH004,2024-02-17,Assessment intervention,Targeted intervention for students requiring academic support,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,c12be7c828376a224f63008ca67d8ed52ba8423215a23fca7b1591ebf59845da,2026-06-08T04:19:38.275Z
EVT00008,SCH004,2024-04-19,Attendance campaign,Campaign to improve student attendance and reduce unexplained absences,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,4dc859d263156e7dfd5a45ac920904b4401ecde0b10b35353cad800a663d2770,2026-06-08T04:19:38.275Z
EVT00009,SCH004,2024-04-12,Assessment intervention,Targeted intervention for students requiring academic support,2025-01-15,dev_education_qa_pipeline_2025-01-15_initial_20260608T041742Z,2026-06-08T04:18:00.331Z,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/school_events/batch_id=2025-01-15/school_events.json,c96f24b65b218c18487527b696a06bdc1dea2bf9bdcaeb444e9c01045ea5467e,2026-06-08T04:19:38.275Z
EVT00010,SCH005,2024-09-13,Wellbeing pr

In [0]:
spark.table(f"{catalog}.silver.school_events").printSchema()

(
    spark.table(f"{catalog}.silver.school_events")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
    .show()
)

root
 |-- event_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+----------+-----+
|  batch_id|count|
+----------+-----+
|2025-01-15|  146|
|2026-01-15|  135|
+----------+-----+



In [0]:
# Validation

silver_tables = [
    "schools",
    "students",
    "attendance",
    "assessment_results",
    "school_events"
]

count_dfs = []

for table_name in silver_tables:
    count_df = (
        spark.table(f"{catalog}.silver.{table_name}")
        .groupBy("batch_id")
        .count()
        .withColumn("table_name", F.lit(table_name))
        .select("table_name", "batch_id", "count")
    )

    count_dfs.append(count_df)

silver_count_summary = reduce(
    lambda df1, df2: df1.unionByName(df2),
    count_dfs
)

display(
    silver_count_summary
    .orderBy("table_name", "batch_id")
)

table_name,batch_id,count
assessment_results,2025-01-15,30001
assessment_results,2026-01-15,24235
attendance,2025-01-15,120003
attendance,2026-01-15,96939
school_events,2025-01-15,146
school_events,2026-01-15,135
schools,2025-01-15,50
schools,2026-01-15,51
students,2025-01-15,10002
students,2026-01-15,10502


### Reconciliation

In [0]:
tables = [
    "schools",
    "students",
    "attendance",
    "assessment_results",
    "school_events"
]

reconciliation_dfs = []


In [0]:
for table_name in tables:
    bronze_counts = (
        spark.table(f"{catalog}.bronze.{table_name}")
        .groupBy("batch_id")
        .count()
        .withColumnRenamed("count", "bronze_count")
    )

    silver_counts = (
        spark.table(f"{catalog}.silver.{table_name}")
        .groupBy("batch_id")
        .count()
        .withColumnRenamed("count", "silver_count")
    )

    reconciliation_df = (
        bronze_counts
        .join(silver_counts, on="batch_id", how="full")
        .withColumn("table_name", F.lit(table_name))
        .withColumn("reconciliation_timestamp", F.current_timestamp())
        .withColumn(
            "reconciliation_status",
            F.when(F.col("bronze_count") == F.col("silver_count"), F.lit("PASS"))
                .otherwise(F.lit("FAIL"))
        )
        .select(
            "table_name",
            "batch_id",
            "bronze_count",
            "silver_count",
            "reconciliation_status",
            "reconciliation_timestamp"
        )
    )

    reconciliation_dfs.append(reconciliation_df)

row_count_reconciliation_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    reconciliation_dfs
)

target_path = f"{lake_root}/qa/row_count_reconciliation"

(
    row_count_reconciliation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", target_path)
    .saveAsTable(f"{catalog}.qa.row_count_reconciliation")
)

display(
    spark.table(f"{catalog}.qa.row_count_reconciliation")
    .orderBy("table_name", "batch_id")
)

table_name,batch_id,bronze_count,silver_count,reconciliation_status,reconciliation_timestamp
assessment_results,2025-01-15,30001,30001,PASS,2026-06-08T04:51:29.524Z
assessment_results,2026-01-15,24235,24235,PASS,2026-06-08T04:51:29.524Z
attendance,2025-01-15,120003,120003,PASS,2026-06-08T04:51:29.524Z
attendance,2026-01-15,96939,96939,PASS,2026-06-08T04:51:29.524Z
school_events,2025-01-15,146,146,PASS,2026-06-08T04:51:29.524Z
school_events,2026-01-15,135,135,PASS,2026-06-08T04:51:29.524Z
schools,2025-01-15,50,50,PASS,2026-06-08T04:51:29.524Z
schools,2026-01-15,51,51,PASS,2026-06-08T04:51:29.524Z
students,2025-01-15,10002,10002,PASS,2026-06-08T04:51:29.524Z
students,2026-01-15,10502,10502,PASS,2026-06-08T04:51:29.524Z


In [0]:
# Summary

silver_type_checks = []

checks = [
    {
        "table_name": "schools",
        "checks": [
            ("open_date", "invalid_or_missing_open_date")
        ]
    },
    {
        "table_name": "students",
        "checks": [
            ("year_level", "invalid_or_missing_year_level"),
            ("enrolment_date", "invalid_or_missing_enrolment_date")
        ]
    },
    {
        "table_name": "attendance",
        "checks": [
            ("attendance_month", "invalid_or_missing_attendance_month"),
            ("possible_days", "invalid_or_missing_possible_days"),
            ("attended_days", "invalid_or_missing_attended_days")
        ]
    },
    {
        "table_name": "assessment_results",
        "checks": [
            ("assessment_year", "invalid_or_missing_assessment_year"),
            ("score", "invalid_or_missing_score")
        ]
    },
    {
        "table_name": "school_events",
        "checks": [
            ("event_date", "invalid_or_missing_event_date")
        ]
    }
]

for item in checks:
    table_name = item["table_name"]
    df = spark.table(f"{catalog}.silver.{table_name}")

    aggregations = [
        F.count("*").alias("total_records")
    ]

    for column_name, metric_name in item["checks"]:
        aggregations.append(
            F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(metric_name)
        )

    check_df = (
        df.groupBy("batch_id")
        .agg(*aggregations)
        .withColumn("table_name", F.lit(table_name))
    )        

    silver_type_checks.append(check_df)

silver_type_validation = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    silver_type_checks
)

In [0]:
display(
    silver_type_validation
    .select("table_name", "batch_id", "total_records",
            *[c for c in silver_type_validation.columns if c not in ["table_name", "batch_id", "total_records"]])
    .orderBy("table_name", "batch_id")
)

table_name,batch_id,total_records,invalid_or_missing_open_date,invalid_or_missing_year_level,invalid_or_missing_enrolment_date,invalid_or_missing_attendance_month,invalid_or_missing_possible_days,invalid_or_missing_attended_days,invalid_or_missing_assessment_year,invalid_or_missing_score,invalid_or_missing_event_date
assessment_results,2025-01-15,30001,null,null,null,null,null,null,0,0,null
assessment_results,2026-01-15,24235,null,null,null,null,null,null,0,0,null
attendance,2025-01-15,120003,null,null,null,0,0,0,null,null,null
attendance,2026-01-15,96939,null,null,null,0,0,0,null,null,null
school_events,2025-01-15,146,null,null,null,null,null,null,null,null,0
school_events,2026-01-15,135,null,null,null,null,null,null,null,null,0
schools,2025-01-15,50,0,null,null,null,null,null,null,null,null
schools,2026-01-15,51,0,null,null,null,null,null,null,null,null
students,2025-01-15,10002,null,0,0,null,null,null,null,null,null
students,2026-01-15,10502,null,0,0,null,null,null,null,null,null
